# SI100B Fall 2026 · Recitation 6B 配套代码

这个 notebook 配合 HW6 B 班课件使用，包含三部分：

1. **PA2 四道题的参考实现与测试**：按 T1–T4 顺序，先看小例子，再看代码。
2. **Lec11 / Lec12 的类与继承示例**：每个例子都可以直接运行，观察输出。
3. **Midterm 易错样例**：alias、复制、排序、作用域、异常、字典、递归等。

> 代码用于课堂讲解和自测，不要直接把答案提交到 OJ。先自己写一遍，再对照检查。


## 0. 运行方式

在 Jupyter 中从上到下运行单元格即可。代码只使用 Python 标准库。  
如果某段代码故意触发异常，会用 `try/except` 包起来，并打印异常类型。


## 1. PA2 · T1：函数、条件与循环

先把题目拆成四个小函数：

- `is_even`：直接返回布尔表达式；
- `larger`：用 `if/else` 返回较大的数；
- `sum_odd`：闭区间 `[a, b]` 内累加奇数；
- `count_divisible`：数一数 `[a, b]` 里有多少个数能被 `d` 整除。

注意 `range(a, b + 1)` 的 `+1`，漏掉就会少算右端点。


In [ ]:
def is_even(n: int) -> bool:
    return n % 2 == 0


def larger(x: int, y: int) -> int:
    if x > y:
        return x
    return y


def sum_odd(a: int, b: int) -> int:
    total = 0
    for k in range(a, b + 1):
        if k % 2 == 1:
            total += k
    return total


def count_divisible(a: int, b: int, d: int) -> int:
    cnt = 0
    for k in range(a, b + 1):
        if k % d == 0:
            cnt += 1
    return cnt

assert is_even(10) is True
assert is_even(7) is False
assert larger(3, 8) == 8
assert sum_odd(2, 7) == 3 + 5 + 7
assert count_divisible(1, 10, 3) == 3

print("T1 tests passed")
print(is_even(10), larger(3, 8), sum_odd(2, 7), count_divisible(1, 10, 3))

**易错点**

- `return` 写进循环体，第一次迭代就返回，后面的数没算到。
- `sum_odd` 的累加器忘记初始化为 `0`。
- `range` 的右端点不包含，闭区间要写 `b + 1`。
- 函数没有显式 `return` 时，结果是 `None`，不是 `0`。


## 2. PA2 · T2：列表查询、构造新列表、原地修改

这题最重要的是分清三件事：

1. **读取**：`stock[index]` 只取值，不改列表；
2. **构造新列表**：`low_stock` 返回的是**下标**，不是库存值；
3. **原地修改**：`restock` 直接改传入的列表，没有返回值。


In [ ]:
def get_stock(stock: list[int], index: int) -> int:
    return stock[index]


def total_stock(stock: list[int]) -> int:
    total = 0
    for inv in stock:
        total += inv
    return total


def low_stock(stock: list[int], threshold: int) -> list[int]:
    res = []
    for i in range(len(stock)):
        if stock[i] < threshold:
            res.append(i)
    return res


def restock(stock: list[int], amount: int, limit: int) -> None:
    for i in range(len(stock)):
        stock[i] = min(stock[i] + amount, limit)


stock = [3, 8, 2]
before = stock[:]

assert get_stock(stock, 1) == 8
assert total_stock(stock) == 13
assert stock == before, "查询和求和不能改变原列表"
assert low_stock(stock, 5) == [0, 2]
assert stock == before, "low_stock 构造新列表，不能改原列表"

result = restock(stock, 2, 9)
assert result is None
assert stock == [5, 9, 4]

print("T2 tests passed")
print(stock)

**易错点**

- `low_stock` 返回库存值而不是下标；题目要的是位置。
- `restock` 里写了 `stock[i] = ...`，最后又 `return stock`；题目要求原地修改并返回 `None`。
- `min` 的参数顺序别写反：`min(库存 + amount, limit)` 才是封顶。
- `append` 是加到新列表；不要一边遍历 `stock` 一边 `remove`。


## 3. PA2 · T3：打印站的状态、TICK 和 SNAP

先把状态列出来：

- `queues`：每台打印机一个队列；
- `time`：当前时间；
- `completed`：已经完成的任务；
- `snapshots`：某一时刻保存的队列副本。

TICK 的关键顺序是：**先加时间，再让每台打印机各处理一次当前队首**。  
同一秒内，一台打印机只处理一个任务；下一个任务要等下一秒。


In [ ]:
def take_job(queues, job_id):
    for queue in queues:
        for position in range(len(queue)):
            if queue[position][0] == job_id:
                return queue.pop(position)


def simulate(n, commands):
    queues = [[] for _ in range(n)]
    time = 0
    completed = []
    snapshots = []

    for command in commands:
        operation = command[0]

        if operation == "ADD":
            job_id, pages, printer = command[1:]
            queues[printer].append((job_id, pages))

        elif operation == "MOVE":
            job_id, printer, position = command[1:]
            job = take_job(queues, job_id)
            queues[printer].insert(position, job)

        elif operation == "CANCEL":
            take_job(queues, command[1])

        elif operation == "TICK":
            for _ in range(command[1]):
                time += 1
                for printer in range(n):
                    queue = queues[printer]
                    if not queue:
                        continue
                    job_id, pages = queue[0]
                    if pages == 1:
                        queue.pop(0)
                        completed.append((time, job_id, printer))
                    else:
                        queue[0] = (job_id, pages - 1)

        elif operation == "SNAP":
            saved_queues = [queue[:] for queue in queues]
            snapshots.append((time, saved_queues))

    return time, queues, completed, snapshots


# 两台打印机，任务 1 需要 2 秒，任务 2 需要 1 秒
result = simulate(2, [
    ["ADD", 1, 2, 0],
    ["ADD", 2, 1, 0],
    ["TICK", 1],
    ["SNAP"],
    ["TICK", 1],
])

time, queues, completed, snapshots = result
assert time == 2
assert queues == [[(2, 1)], []]
assert completed == [(2, 1, 0)]
assert snapshots[0][0] == 1
assert snapshots[0][1] == [[(1, 1), (2, 1)], []]

print("T3 TICK/SNAP tests passed")
print("time =", time)
print("queues =", queues)
print("completed =", completed)
print("snapshots =", snapshots)

In [ ]:
# MOVE：先移除，再插入；position 按移除后的队列计算
result = simulate(1, [
    ["ADD", 10, 2, 0],
    ["ADD", 20, 1, 0],
    ["MOVE", 20, 0, 0],
    ["TICK", 1],
])
time, queues, completed, _ = result
assert queues == [[(10, 2)]]
assert completed == [(1, 20, 0)]
print("T3 MOVE test passed:", result)

**SNAP 为什么只复制队列就够？**

这里每个任务是 `(job_id, pages)` 的 tuple。tuple 不可变，减页时我们是**换一个新的 tuple**，不是修改旧 tuple。  
所以把每支队列 `queue[:]` 复制出来，快照就不会被后续操作影响。

如果任务本身用 list 存，比如 `[job_id, pages]`，那只复制队列还不够，需要逐个任务复制，或者用 `copy.deepcopy`。


In [ ]:
from copy import deepcopy

queues = [[[1, 2]], []]
snapshot = [[job[:] for job in queue] for queue in queues]
queues[0][0][1] -= 1

assert snapshot == [[[1, 2]], []]
assert queues == [[[1, 1]], []]

queues2 = [[[1, 2]], []]
snapshot2 = deepcopy(queues2)
queues2[0][0][1] -= 1
assert snapshot2 == [[[1, 2]], []]
print("snapshot examples passed")

**T3 易错点**

- TICK 里先处理打印机、后加时间：完成时间会错。
- 同一秒让一台打印机连续清空多个任务：任务 2 的完成时间会提前。
- MOVE 在移除前算 position：同队列移动时位置会偏一位。
- CANCEL 把任务记进 `completed`：题目只说取消，不算完成。
- SNAP 只保存 `queues` 的引用：后续 TICK 会把快照一起改掉。


## 4. PA2 · T4：商品、切片、排序和交易历史

商品是 tuple：`(code, name, price, stock)`。tuple 不能原地改，所以补货、调价、购买成功都要**解包后换一个新 tuple**。

切片查询和调价都作用在**当前顺序**上；`reorder` 会改变顺序，后面的 `inventory` 结果也会跟着变。


In [ ]:
Product = tuple[str, str, float, int]
Sale = tuple[str, int, float]


def parse_item(line: str) -> Product:
    code, name, price, stock = line.split(",")
    return code, name, float(price), int(stock)


def find_item_index(items, code):
    for i in range(len(items)):
        if items[i][0] == code:
            return i
    raise ValueError("Unknown product code.")


def buy(items, sales, code, quantity):
    index = find_item_index(items, code)
    item_code, name, price, stock = items[index]
    if stock < quantity:
        return None
    cost = price * quantity
    items[index] = (item_code, name, price, stock - quantity)
    sales.append((code, quantity, cost))
    return cost


def restock(items, code, quantity):
    index = find_item_index(items, code)
    item_code, name, price, stock = items[index]
    items[index] = (item_code, name, price, stock + quantity)


def adjust_prices(items, rule, value, start, end, step):
    indices = range(len(items))[start:end:step]
    for i in indices:
        code, name, old_price, stock = items[i]
        if rule == "discount":
            new_price = old_price * (1 - value / 100)
        elif rule == "markup":
            new_price = old_price * (1 + value / 100)
        else:
            new_price = value
        items[i] = (code, name, new_price, stock)


def price_asc_key(item):
    return item[2], item[0]


def price_desc_key(item):
    return -item[2], item[0]


def reorder(items, key, order):
    if key == "price":
        items.sort(key=price_asc_key if order == "ASC" else price_desc_key)
    else:
        def stock_asc(item):
            return item[3], item[0]
        def stock_desc(item):
            return -item[3], item[0]
        items.sort(key=stock_asc if order == "ASC" else stock_desc)


def inventory_codes(items, start, end, step):
    return [item[0] for item in items[start:end:step]]


def inventory_value(items, start, end, step):
    selected = items[start:end:step]
    if not selected:
        return None
    return sum(price * stock for _, _, price, stock in selected)


def units_sold(sales, code):
    return sum(quantity for item_code, quantity, _ in sales if item_code == code)


def total_revenue(sales):
    return sum(cost for _, _, cost in sales)


items = [
    parse_item("A,Apple,2.50,10"),
    parse_item("B,Banana,1.00,4"),
]
sales = []

assert buy(items, sales, "A", 3) == 7.50
assert items[0][3] == 7
assert buy(items, sales, "B", 9) is None
assert items[1][3] == 4, "失败购买不能改库存"
assert sales == [("A", 3, 7.50)], "失败购买不能写历史"

restock(items, "B", 3)
adjust_prices(items, "discount", 10, 0, 2, 1)
reorder(items, "price", "DESC")

assert inventory_codes(items, 0, 2, 1) == ["A", "B"]
assert round(inventory_value(items, 0, 2, 1), 2) == 22.05
assert units_sold(sales, "A") == 3
assert total_revenue(sales) == 7.50

print("T4 tests passed")
print(items)
print(sales)

**T4 易错点**

- 直接 `items[i][2] = ...`：tuple 不支持，必须换整个 tuple。
- 降序用 `reverse=True`：价格降了，但相同价格的 code 也反了；题目要求 code 仍然升序，所以用 `(-price, code)`。
- `adjust` 的 `start/end/step` 按当前顺序算；前面 `reorder` 过，后面的结果会变。
- 购买失败时先改了库存再判断：库存和历史都被污染。
- `revenue` 用当前价格重算：应该用销售时记录的 `cost`，调价不能改写历史。


## 5. Lec11：类、实例与 dunder 方法

`class` 定义一种类型；实例保存自己的数据。`self` 指向正在操作的那个实例。  
`c.distance(origin)` 等价于 `Coordinate.distance(c, origin)`，只是点号写法更自然。


In [ ]:
class Coordinate:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def distance(self, other):
        dx = self.x - other.x
        dy = self.y - other.y
        return (dx ** 2 + dy ** 2) ** 0.5

    def __str__(self):
        return f"<{self.x},{self.y}>"


c = Coordinate(3, 4)
origin = Coordinate(0, 0)
assert c.distance(origin) == 5.0
assert str(c) == "<3,4>"
print(c, c.distance(origin))

In [ ]:
class Fraction:
    def __init__(self, num, denom):
        self.num = num
        self.denom = denom

    def __str__(self):
        return f"{self.num}/{self.denom}"

    def __mul__(self, other):
        if isinstance(other, Fraction):
            return Fraction(self.num * other.num, self.denom * other.denom)
        if isinstance(other, int):
            return Fraction(self.num * other, self.denom)
        raise TypeError("unsupported operand")

    def __float__(self):
        return self.num / self.denom


a = Fraction(1, 2)
b = Fraction(2, 3)
c = a * b
assert str(c) == "2/6"
assert abs(float(c) - 1 / 3) < 1e-12
print(c, float(c))
print(a * 3)

In [ ]:
class SimpleFraction:
    def __init__(self, num, denom):
        self.num = num
        self.denom = denom

    def get_inverse(self):
        return self.denom / self.num

    def invert(self):
        self.num, self.denom = self.denom, self.num
        return None


f = SimpleFraction(3, 4)
assert f.get_inverse() == 4 / 3
assert f.invert() is None
assert (f.num, f.denom) == (4, 3)
print("return vs mutate examples passed")

**Lec11 易错点**

- 方法定义忘写 `self`，或者调用时又手动传了一次 `self`。
- `__init__` 里写 `x = xval` 而不是 `self.x = xval`，实例没有保存属性。
- `__str__` 必须返回字符串；返回数字会在 `print` 时报 `TypeError`。
- `isinstance(x, C)` 能兼容子类；`type(x) == C` 不能。
- `get_inverse()` 返回一个值；`invert()` 改自身并返回 `None`，别把两件事混在一起。


### 5.5 Lec12：getter/setter 与接口

Python 不强制 private，但 getter/setter 能给外部一个稳定入口。setter 可以校验，直接赋值会绕过校验。


In [ ]:
class Workout:
    def __init__(self, start, end, cal=200):
        self.start, self.end = start, end
        self.calories = cal

    def get_calories(self):
        return self.calories

    def set_calories(self, c):
        if c < 0:
            raise ValueError("calories < 0")
        self.calories = c

    def __str__(self):
        return f"<{self.get_calories()} kcal>"


w = Workout("9/30 1:35 PM", "9/30 2:05 PM")
w.set_calories(260)
assert w.get_calories() == 260

try:
    w.set_calories(-5)
except ValueError as error:
    print(type(error).__name__, error)

w.calories = -5          # 绕过 setter，没有校验
assert w.get_calories() == -5
print("getter/setter example passed")

## 6. Lec12：继承、super 与方法查找

子类继承父类的数据和行为，可以新增方法，也可以覆盖同名方法。  
子类 `__init__` 先调用 `super().__init__(...)`，再初始化自己特有的属性。


In [ ]:
class Workout:
    cal_per_hr = 200

    def __init__(self, start, end, calories=None):
        self.start = start
        self.end = end
        self.calories = calories

    def kind(self):
        return "Workout"

    def get_calories(self):
        if self.calories is not None:
            return self.calories
        return self.cal_per_hr

    def __str__(self):
        return f"{self.kind()}: {self.get_calories()} cal"


class RunWorkout(Workout):
    def __init__(self, start, end, elev=0, calories=None):
        super().__init__(start, end, calories)
        self.elev = elev

    def kind(self):
        return "Run"

    def get_elev(self):
        return self.elev

    def get_calories(self):
        return super().get_calories() + self.elev * 2


w = Workout("9:00", "10:00")
rw = RunWorkout("9:00", "10:00", elev=100)
assert w.kind() == "Workout"
assert rw.kind() == "Run"
assert rw.get_calories() == 400
assert isinstance(rw, Workout) is True
assert isinstance(w, RunWorkout) is False
print(w)
print(rw)
print(rw.get_elev())

In [ ]:
# 类变量由全类共享；通过实例赋值会新建实例属性
Workout.cal_per_hr = 250
assert w.cal_per_hr == 250
assert rw.cal_per_hr == 250

rw.cal_per_hr = 999
assert rw.cal_per_hr == 999
assert Workout.cal_per_hr == 250
assert w.cal_per_hr == 250
print("class variable example passed")

In [ ]:
# 父类实例不能调用子类专有方法
try:
    w.get_elev()
except AttributeError as error:
    print(type(error).__name__, error)

# 覆盖方法中可以用 super() 复用父类逻辑
Workout.cal_per_hr = 200
assert RunWorkout("9:00", "10:00", elev=0).get_calories() == 200
print("method lookup examples passed")

**Lec12 易错点**

- 把类变量当成每个实例独立的一份；`self.cal_per_hr = ...` 会新建实例属性，不会改类变量。
- 子类覆盖 `__init__` 后忘记 `super().__init__`，父类属性没建立。
- 父类实例调用子类专有方法，运行时才报 `AttributeError`。
- 方法查找从当前类开始，找不到再沿父类向上；不是永远调用父类版本。
- 继承不是越深越好；能用组合解决时，不要硬造层次。

> 本次期中范围明确排除 OOP 与 inheritance（Lecture 12, p.4）。这部分用于理解课程内容，不是期中重点。


## 7. Midterm 易错样例：先预测，再运行

下面每段代码都先遮住输出，自己算一遍，再运行。  
重点是理解**对象、名字绑定、返回值**，不是背结论。


In [ ]:
# alias：b 和 a 指向同一个列表
a = [1, 2, 3]
b = a
b.append(4)
assert a == [1, 2, 3, 4]
print("alias:", a, b)

In [ ]:
# 浅复制 vs 深复制
from copy import deepcopy

a = [1, [2]]
shallow = a[:]
deep = deepcopy(a)
a[1].append(3)

print("original:", a)
print("shallow :", shallow)
print("deep    :", deep)

In [ ]:
# sort() 改原列表并返回 None；sorted() 返回新列表
L = [3, 1, 2]
x = L.sort()
print("L, x:", L, x)

M = [3, 1, 2]
y = sorted(M)
print("M, y:", M, y)

In [ ]:
# 边遍历边删除会跳过元素
L = [1, 2, 2, 3]
for x in L:
    if x == 2:
        L.remove(x)
print("bad remove:", L)

L2 = [1, 2, 2, 3]
for x in L2[:]:
    if x == 2:
        L2.remove(x)
print("safe remove:", L2)

In [ ]:
# 函数内先使用、后赋值同名变量：UnboundLocalError
x = 5

def broken():
    print(x)
    x = 1

try:
    broken()
except UnboundLocalError as error:
    print(type(error).__name__, error)

In [ ]:
# slice 与 range 的 stop 都不包含
s = "0123456"
print(s[1:6:2])
print(s[::-1])
print(list(range(2, 6, 2)))

In [ ]:
# float 不要直接用 == 比较
a = 0.1 + 0.2
b = 0.3
print(a, b, a == b)
print(abs(a - b) < 1e-10)

In [ ]:
# dict：in 默认检查 key
d = {"a": 1}
print("a" in d, 1 in d)
print(d.get("b", 0))
print(list(d.items()))

In [ ]:
# try/except/else/finally 的执行顺序
def demo(raw):
    try:
        value = 10 / int(raw)
    except ValueError:
        print("except")
    else:
        print("else", value)
    finally:
        print("finally")

demo("2")
demo("x")
try:
    demo("0")
except ZeroDivisionError:
    print("outer caught ZeroDivisionError")

In [ ]:
# 递归：base case、规模缩小、把结果 return 回去
def fact(n):
    if n == 0:
        return 1
    return n * fact(n - 1)

def bad_fact(n):
    if n == 0:
        return 1
    bad_fact(n - 1)   # 没有 return，结果不会传回上一层

print(fact(5))
print(bad_fact(5))

In [ ]:
# 手推：tuple 不可变、交换和 *args
def mean(*args):
    return sum(args) / len(args)

t = (1, 2, 3)
a, b = 10, 20
a, b = b, a
print(t[1:], mean(*t), a, b)

## 8. 最后检查清单

- 每个函数是否在**所有路径**都返回了正确的值？
- 题目要求原地修改还是返回新对象？
- 有没有把 `print` 当成 `return`？
- 循环的区间、切片的 stop、`range` 的 step 是否清楚？
- 复制的是对象本身，还是只复制了外层容器？
- 递归有没有 base case，参数是否每次都在靠近 base case？
- 异常类型是否精确？`raise ValueError` 和 `return "ValueError"` 完全不是一回事。
- OJ 提交前删掉调试 `print`，不要输出题目没要求的提示语。
